In [1]:
from set_seed_utils import set_random_seed
import os
import random
import numpy as np
import pickle
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from token_utils_rep import EHRTokenizer
from dataset_utils_rep import HBERTFinetuneEHRDataset, batcher, UniqueIDSampler
from HEART_rep import HBERT_Finetune
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve, precision_recall_fscore_support
import pandas as pd

Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
PHENO_ORDER = [
    "Acute and unspecified renal failure",
    "Acute cerebrovascular disease",
    "Acute myocardial infarction",
    "Cardiac dysrhythmias",
    "Chronic kidney disease",
    "Chronic obstructive pulmonary disease",
    "Conduction disorders",
    "Congestive heart failure; nonhypertensive",
    "Coronary atherosclerosis and related",
    "Disorders of lipid metabolism",
    "Essential hypertension",
    "Fluid and electrolyte disorders",
    "Gastrointestinal hemorrhage",
    "Hypertension with complications",
    "Other liver diseases",
    "Other lower respiratory disease",
    "Pneumonia",
    "Septicemia (except in labor)",
]

In [4]:
@torch.no_grad()
def evaluate(model, 
             dataloader, 
             device, 
             long_seq_idx=None, 
             task_type="binary", 
             subgroup_labels=None):
    """
    subgroup_labels: None 或 pandas.DataFrame / Series，长度必须等于 dataloader 总样本数，
                     每列为一个 0/1 subgroup（如 DIABETES/HF/...），仅在 binary 任务下使用。
    返回：
        all_performance:     overall 指标
        subset_performance:  long_seq 子集指标（若 long_seq_idx 不为 None，否则为 None）
        subgroup_performance: dict[subgroup_name -> metrics_dict] 或 None
    """
    model.eval()
    predicted_scores, gt_labels = [], []

    # 推理：收集 logits 与 labels
    for _, batch in enumerate(tqdm(dataloader, desc="Running inference")):
        batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
        labels = batch[-1]
        output_logits = model(*batch[:-1])
        predicted_scores.append(output_logits)
        gt_labels.append(labels)

    # ============= 二分类任务 ============= #
    if task_type == "binary":
        logits_all = torch.cat(predicted_scores, dim=0).view(-1)           # [N]
        labels_all = torch.cat(gt_labels, dim=0).view(-1).cpu().numpy()    # [N]
        scores_all = logits_all.cpu().numpy()
        ypred_all  = (logits_all > 0).float().cpu().numpy()

        tp = (ypred_all * labels_all).sum()
        precision = tp / (ypred_all.sum() + 1e-8)
        recall    = tp / (labels_all.sum() + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        roc_auc   = roc_auc_score(labels_all, scores_all)
        prec_curve, rec_curve, _ = precision_recall_curve(labels_all, scores_all)
        pr_auc    = auc(rec_curve, prec_curve)

        all_performance = {
            "precision": float(precision),
            "recall":    float(recall),
            "f1":        float(f1),
            "auc":       float(roc_auc),
            "prauc":     float(pr_auc),
        }

        # ---- long_seq 子集 ----
        subset_performance = None
        if long_seq_idx is not None:
            idx = torch.as_tensor(long_seq_idx, device=logits_all.device, dtype=torch.long)
            logits_sub = logits_all.index_select(0, idx).view(-1)
            labels_sub = torch.as_tensor(labels_all, device=logits_all.device)[idx].cpu().numpy()
            scores_sub = logits_sub.cpu().numpy()
            ypred_sub  = (logits_sub > 0).float().cpu().numpy()

            tp = (ypred_sub * labels_sub).sum()
            precision = tp / (ypred_sub.sum() + 1e-8)
            recall    = tp / (labels_sub.sum() + 1e-8)
            f1        = 2 * precision * recall / (precision + recall + 1e-8)
            roc_auc   = roc_auc_score(labels_sub, scores_sub)
            prec_curve, rec_curve, _ = precision_recall_curve(labels_sub, scores_sub)
            pr_auc    = auc(rec_curve, prec_curve)

            subset_performance = {
                "precision": float(precision),
                "recall":    float(recall),
                "f1":        float(f1),
                "auc":       float(roc_auc),
                "prauc":     float(pr_auc),
            }

        # ---- subgroup analysis（仅 binary）----
        subgroup_performance = None
        if subgroup_labels is not None:
            import pandas as pd
            subgroup_performance = {}

            if isinstance(subgroup_labels, pd.Series):
                subgroup_df = subgroup_labels.to_frame()
            else:
                subgroup_df = subgroup_labels

            if len(subgroup_df) != logits_all.shape[0]:
                raise ValueError(
                    f"subgroup_labels 行数 {len(subgroup_df)} 与样本数 {logits_all.shape[0]} 不一致"
                )

            for col in subgroup_df.columns:
                mask_np = subgroup_df[col].to_numpy().astype(bool)
                if mask_np.sum() == 0:
                    continue  # 这个 subgroup 没有样本，跳过

                idx = torch.as_tensor(
                    np.where(mask_np)[0],
                    device=logits_all.device,
                    dtype=torch.long,
                )

                logits_sub = logits_all.index_select(0, idx).view(-1)
                labels_sub = torch.as_tensor(labels_all, device=logits_all.device)[idx].cpu().numpy()
                scores_sub = logits_sub.cpu().numpy()
                ypred_sub  = (logits_sub > 0).float().cpu().numpy()

                tp = (ypred_sub * labels_sub).sum()
                precision = tp / (ypred_sub.sum() + 1e-8)
                recall    = tp / (labels_sub.sum() + 1e-8)
                f1        = 2 * precision * recall / (precision + recall + 1e-8)
                roc_auc   = roc_auc_score(labels_sub, scores_sub)
                prec_curve, rec_curve, _ = precision_recall_curve(labels_sub, scores_sub)
                pr_auc    = auc(rec_curve, prec_curve)

                subgroup_performance[col] = {
                    "precision": float(precision),
                    "recall":    float(recall),
                    "f1":        float(f1),
                    "auc":       float(roc_auc),
                    "prauc":     float(pr_auc),
                }

        return all_performance, subset_performance, subgroup_performance

    # ============= Multi-label 任务 ============= #
    else:
        logits_all = torch.cat(predicted_scores, dim=0)    # [B, C]
        labels_all_t = torch.cat(gt_labels, dim=0)         # [B, C]

        def _compute_metrics(logits_sub, labels_sub):
            if logits_sub.device.type == "cpu" and logits_sub.dtype == torch.float16:
                prob_t = torch.sigmoid(logits_sub.float())
            else:
                prob_t = torch.sigmoid(logits_sub)

            ypred_t = (logits_sub > 0).to(torch.int32)

            y_true = labels_sub.cpu().numpy().astype(np.int32)
            y_pred = ypred_t.cpu().numpy().astype(np.int32)
            scores = prob_t.cpu().numpy()

            p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(
                y_true, y_pred, average=None, zero_division=0
            )

            C = y_true.shape[1]
            aucs, praucs = [], []
            for c in range(C):
                yt, ys = y_true[:, c], scores[:, c]
                if yt.max() == yt.min():
                    aucs.append(np.nan)
                    praucs.append(np.nan)
                else:
                    aucs.append(roc_auc_score(yt, ys))
                    prec_curve, rec_curve, _ = precision_recall_curve(yt, ys)
                    praucs.append(auc(rec_curve, prec_curve))

            summary = {
                "precision": float(np.mean(p_cls)),
                "recall":    float(np.mean(r_cls)),
                "f1":        float(np.mean(f1_cls)),
                "auc":       float(np.nanmean(aucs)) if np.any(~np.isnan(aucs)) else float("nan"),
                "prauc":     float(np.nanmean(praucs)) if np.any(~np.isnan(praucs)) else float("nan"),
            }

            per_class_df = pd.DataFrame({
                "precision": p_cls,
                "recall":    r_cls,
                "f1":        f1_cls,
                "auc":       aucs,
                "prauc":     praucs,
            }, index=PHENO_ORDER)

            return {"global": summary, "per_class": per_class_df}

        all_performance = _compute_metrics(logits_all, labels_all_t)

        subset_performance = None
        if long_seq_idx is not None:
            idx = torch.as_tensor(long_seq_idx, device=logits_all.device, dtype=torch.long)
            subset_performance = _compute_metrics(
                logits_all.index_select(0, idx),
                labels_all_t.index_select(0, idx)
            )

        # multi-label 不做 subgroup，统一返回 None
        subgroup_performance = None
        return all_performance, subset_performance, subgroup_performance

In [5]:
args = {
    "seed": 0,
    "dataset": "MIMIC-III", 
    "task": "readmission",  # options: death, stay, readmission, next_diag_6m, next_diag_12m
    "encoder": "hi",  # options: hi_edge, hi_node, hi_edge_node
    "batch_size": 4,
    "eval_batch_size": 4,
    "pretrain_mask_rate": 0.7,
    "lr": 1e-4,
    "epochs": 500,
    "num_hidden_layers": 5,
    "num_attention_heads": 6,
    "attention_probs_dropout_prob": 0.2,
    "hidden_dropout_prob": 0.2,
    "edge_hidden_size": 32,
    "hidden_size": 288,  # must be divisible by num_attention_heads
    "intermediate_size": 288,
    "save_model": True,
    "gat": "None",
    "gnn_n_heads": 1,
    "gnn_temp": 1,
    "diag_med_emb": "simple",  # simple, tree
    "early_stop_patience": 5,
}

In [6]:
exp_name = "Pretrain-HBERT" \
    + "-" + str(args["dataset"]) \
    + "-" + str(args["encoder"]) \
    + "-" + str(args["pretrain_mask_rate"]) \
    + "-" + str(args["hidden_size"]) \
    + "-" + str(args["edge_hidden_size"]) \
    + "-" + str(args["num_hidden_layers"]) \
    + "-" + str(args["num_attention_heads"]) \
    + "-" + str(args["attention_probs_dropout_prob"]) \
    + "-" + str(args["hidden_dropout_prob"]) \
    + "-" + str(args["intermediate_size"]) \
    + "-" + str(args["gat"]) \
    + "-" + str(args["gnn_n_heads"]) \
    + "-" + str(args["gnn_temp"]) \
    + "-" + str(args["diag_med_emb"])
print(exp_name)

Pretrain-HBERT-MIMIC-III-hi-0.7-288-32-5-6-0.2-0.2-288-None-1-1-simple


In [7]:
pretrained_weight_path = "./pretrained_models/" + exp_name + f"/pretrained_model.pt"
finetune_exp_name = f"Finetune-{args['task']}-" + exp_name
save_path = "./saved_model/" + finetune_exp_name
if args["save_model"] and not os.path.exists(save_path):
    os.makedirs(save_path)

In [8]:
args["predicted_token_type"] = ["diag"]
args["special_tokens"] = ("[PAD]", "[CLS]", "[SEP]", "[MASK0]")
args["max_visit_size"] = 15

full_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic.pkl"

if args["task"] == "next_diag_6m":
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_nextdiag_6m.pkl"
elif args["task"] == "next_diag_12m":
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_nextdiag_12m.pkl"
else:
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_downstream.pkl"

In [9]:
ehr_data = pickle.load(open(full_data_path, 'rb'))
diag_sentences = ehr_data["ICD9_CODE"].values.tolist()
gender_set = [["M"], ["F"]]
age_gender_set = [[str(c) + "_" + gender] for c in set(ehr_data["AGE"].values.tolist()) for gender in ["M", "F"]]
age_set = [[c] for c in set(ehr_data["AGE"].values.tolist())]    

In [10]:
tokenizer = EHRTokenizer(diag_sentences, gender_set, age_set, age_gender_set, special_tokens=args["special_tokens"])

In [11]:
train_data, val_data, test_data = pickle.load(open(finetune_data_path, 'rb'))

subgroup_names = ["DIABETES", "HYPERTENSION", "CKD", "HEART_FAILURE", "CAD", "COPD", "LIVER_DISEASE", "CANCER"]
val_subgroup_labels = val_data[subgroup_names].copy()
test_subgroup_labels = test_data[subgroup_names].copy()

In [12]:
train_dataset = HBERTFinetuneEHRDataset(
    train_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

val_dataset = HBERTFinetuneEHRDataset(
    val_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

test_dataset = HBERTFinetuneEHRDataset(
    test_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

train_dataloader = DataLoader(
    train_dataset, 
    batch_sampler=UniqueIDSampler(train_dataset.get_ids(), batch_size=args["batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False), 
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_sampler=UniqueIDSampler(val_dataset.get_ids(), batch_size=args["batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False), 
)

test_dataloader = DataLoader(
    test_dataset, 
    batch_sampler=UniqueIDSampler(test_dataset.get_ids(), batch_size=args["eval_batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False),
)

3153 6266 6358


In [13]:
long_adm_seq_crite = 3
val_long_seq_idx, test_long_seq_idx = [], []
for i in range(len(val_dataset)):
    hadm_id = list(val_dataset.records.keys())[i]
    num_adms = len(val_dataset.records[hadm_id])
    if num_adms >= long_adm_seq_crite:
        val_long_seq_idx.append(i)
for i in range(len(test_dataset)):
    hadm_id = list(test_dataset.records.keys())[i]
    num_adms = len(test_dataset.records[hadm_id])
    if num_adms >= long_adm_seq_crite:
        test_long_seq_idx.append(i)
print(len(val_long_seq_idx), len(test_long_seq_idx))

777 861


In [14]:
# examine a batch
batch = next(iter(train_dataloader))  # 取第一个 batch
input_ids, input_types, edge_index, visit_positions, labeled_batch_idx, labels = batch

# 打印每个张量的形状
print("input_ids shape:", input_ids.shape)
print("input_types shape:", input_types.shape)
print("visit_positions shape:", visit_positions.shape)
print("labeled_batch_idx shape:", len(labeled_batch_idx)) # it is a list
print("labels shape:", labels.shape)

input_ids shape: torch.Size([9, 39])
input_types shape: torch.Size([9, 39])
visit_positions shape: torch.Size([9])
labeled_batch_idx shape: 4
labels shape: torch.Size([4, 1])


In [15]:
args["vocab_size"] = len(args["special_tokens"]) + \
                     len(tokenizer.diag_voc.id2word) + \
                     len(tokenizer.age_voc.id2word) + \
                     len(tokenizer.gender_voc.id2word) + \
                     len(tokenizer.age_gender_voc.id2word)
args["label_vocab_size"] = 18  # only for diagnosis

In [16]:
if args["task"] in ["death", "stay", "readmission"]:
    eval_metric = "f1"
    task_type = "binary"
    loss_fn = F.binary_cross_entropy_with_logits
else:
    eval_metric = "prauc"
    task_type = "l2r"
    loss_fn = lambda x, y: F.binary_cross_entropy_with_logits(x, y)

In [17]:
def train_with_early_stopping(model, 
                              train_dataloader, 
                              val_dataloader, 
                              test_dataloader,
                              optimizer, 
                              loss_fn, 
                              device, 
                              args,
                              val_long_seq_idx = None,
                              test_long_seq_idx = None,
                              task_type="binary", 
                              eval_metric="f1",
                              val_subgroup_labels=None,
                              test_subgroup_labels=None):
    best_score = 0.
    best_val_metric = None
    best_test_metric = None
    best_test_long_seq_metric = None
    best_val_subgroup_metrics = None
    best_test_subgroup_metrics = None
    epochs_no_improve = 0

    for epoch in range(1, 1 + args["epochs"]):
        model.train()
        ave_loss = 0.

        for step, batch in enumerate(tqdm(train_dataloader, desc="Training Batches")):
            batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]

            labels = batch[-1].float()
            output_logits = model(*batch[:-1])
            
            loss = loss_fn(output_logits.view(-1), labels.view(-1))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            ave_loss += loss.item()

        ave_loss /= (step + 1)

        # ===== Evaluation（带 subgroup） =====
        val_metric, val_long_seq_metric, val_subgroup_metrics = evaluate(
            model, 
            val_dataloader, 
            device, 
            long_seq_idx=val_long_seq_idx, 
            task_type=task_type,
            subgroup_labels=val_subgroup_labels,
        )
        test_metric, test_long_seq_metric, test_subgroup_metrics = evaluate(
            model, 
            test_dataloader, 
            device, 
            long_seq_idx=test_long_seq_idx, 
            task_type=task_type,
            subgroup_labels=test_subgroup_labels,
        )

        if task_type != "binary":
            val_per_class_df = val_metric["per_class"]
            val_metric = val_metric["global"]
            test_per_class_df = test_metric["per_class"]
            test_metric = test_metric["global"]
            
            if val_long_seq_idx is not None and val_long_seq_metric is not None:
                val_long_seq_per_class_df = val_long_seq_metric["per_class"]
                val_long_seq_metric = val_long_seq_metric["global"]
            if test_long_seq_idx is not None and test_long_seq_metric is not None:
                test_long_seq_per_class_df = test_long_seq_metric["per_class"]
                test_long_seq_metric = test_long_seq_metric["global"]

        # Logging
        print(f"\nEpoch: {epoch:03d}, Average Loss: {ave_loss:.4f}")
        print(f"Validation: {val_metric}")
        print(f"Test:       {test_metric}")

        if test_subgroup_metrics is not None:
            print(f"Test-subgroups:       {test_subgroup_metrics}")
        if test_long_seq_metric is not None:
            print(f"Test-long:            {test_long_seq_metric}")

        # Check for improvement
        current_score = val_metric[eval_metric]
        if current_score > best_score:
            best_score = current_score
            if task_type == "binary":
                best_val_metric = val_metric
                best_test_metric = test_metric
                best_test_long_seq_metric = test_long_seq_metric
            else:
                best_val_metric = {"global": val_metric, "per_class": val_per_class_df}
                best_test_metric = {"global": test_metric, "per_class": test_per_class_df}
                best_test_long_seq_metric = {
                    "global": test_long_seq_metric,
                    "per_class": test_long_seq_per_class_df,
                } if test_long_seq_metric is not None else None

            # 只在 binary 任务下保留 subgroup metrics
            best_val_subgroup_metrics = val_subgroup_metrics if task_type == "binary" else None
            best_test_subgroup_metrics = test_subgroup_metrics if task_type == "binary" else None

            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # Early stopping check
        if epochs_no_improve >= args["early_stop_patience"]:
            print(f"\nEarly stopping triggered after {epoch} epochs "
                  f"(no improvement for {args['early_stop_patience']} epochs).")
            break

    print("\nBest validation performance:")
    print(best_val_metric)
    print("Corresponding test performance:")
    print(best_test_metric)
    if best_test_long_seq_metric is not None:
        print("Corresponding test-long performance:")
        print(best_test_long_seq_metric)
    if best_test_subgroup_metrics is not None:
        print("Corresponding test-subgroup performance:")
        print(best_test_subgroup_metrics)

    return best_test_metric, best_test_long_seq_metric, best_test_subgroup_metrics

In [18]:
random.seed(42)
seeds = [random.randint(0, 2**32 - 1) for _ in range(5)]
print(seeds)

[2746317213, 1181241943, 958682846, 3163119785, 1812140441]


In [19]:
final_metrics, final_long_seq_metrics, final_subgroup_metrics = [], [], []

for seed in seeds:
    args["seed"] = seed
    set_random_seed(args["seed"])
    print(f"Training with seed: {args['seed']}")
    
    # Initialize model, optimizer, and loss function
    model = HBERT_Finetune(args)
    model.load_weight(torch.load(pretrained_weight_path, weights_only=True))
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args["lr"])
    
    best_test_metric, best_test_long_seq_metric, best_test_subgroup_metrics = train_with_early_stopping(
        model, 
        train_dataloader, 
        val_dataloader, 
        test_dataloader,
        optimizer, 
        loss_fn, 
        device, 
        args,
        val_long_seq_idx,
        test_long_seq_idx,
        task_type=task_type,
        val_subgroup_labels=val_subgroup_labels,
        test_subgroup_labels=test_subgroup_labels)
    
    final_metrics.append(best_test_metric)
    final_long_seq_metrics.append(best_test_long_seq_metric)
    final_subgroup_metrics.append(best_test_subgroup_metrics)

[INFO] Random seed set to 2746317213
Training with seed: 2746317213


Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 633.34it/s]



Epoch: 001, Average Loss: 0.6739
Validation: {'precision': 0.5283528352787727, 'recall': 0.2359324758832961, 'f1': 0.32620171843087536, 'auc': 0.6182402370978367, 'prauc': 0.4986640067577649}
Test:       {'precision': 0.5369127516733481, 'recall': 0.24806201550291448, 'f1': 0.3393425195352483, 'auc': 0.6280312170419523, 'prauc': 0.5055248032122508}
Test-subgroups:       {'DIABETES': {'precision': 0.5684754521816932, 'recall': 0.2689486552534358, 'f1': 0.365145223849369, 'auc': 0.6441825534616934, 'prauc': 0.5331318044865809}, 'HYPERTENSION': {'precision': 0.5359877488432467, 'recall': 0.24613220815579373, 'f1': 0.3373493932738406, 'auc': 0.6339423019039332, 'prauc': 0.514210510056649}, 'CKD': {'precision': 0.5670995670750173, 'recall': 0.2614770459029645, 'f1': 0.3579234929382409, 'auc': 0.6409498599367789, 'prauc': 0.5246205179553222}, 'HEART_FAILURE': {'precision': 0.5691906005073318, 'recall': 0.2620192307660815, 'f1': 0.35884773230243366, 'auc': 0.6254876489251489, 'prauc': 0.5236

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 634.41it/s]



Epoch: 002, Average Loss: 0.6292
Validation: {'precision': 0.5554711246158399, 'recall': 0.2938102893878866, 'f1': 0.3843322772812198, 'auc': 0.6398188807599932, 'prauc': 0.519519855344162}
Test:       {'precision': 0.5507544581580881, 'recall': 0.31124031007631303, 'f1': 0.3977216397624671, 'auc': 0.6502799253942655, 'prauc': 0.532715382841997}
Test-subgroups:       {'DIABETES': {'precision': 0.5160599571623969, 'recall': 0.306615776077524, 'f1': 0.384676771056166, 'auc': 0.6284827042513508, 'prauc': 0.4950906625987903}, 'HYPERTENSION': {'precision': 0.5345501955601754, 'recall': 0.29753265602106294, 'f1': 0.3822843776865112, 'auc': 0.6448673025088119, 'prauc': 0.5174049537783106}, 'CKD': {'precision': 0.5279720279535673, 'recall': 0.3192389006275002, 'f1': 0.3978919584023774, 'auc': 0.6351480642450221, 'prauc': 0.49356977256737516}, 'HEART_FAILURE': {'precision': 0.5595505617851786, 'recall': 0.29999999999638555, 'f1': 0.3905882307438924, 'auc': 0.6460818062164624, 'prauc': 0.532566

...


Epoch: 006, Average Loss: 0.5079
Validation: {'precision': 0.48846620801097346, 'recall': 0.4851286173613942, 'f1': 0.4867916868714571, 'auc': 0.6217650439419962, 'prauc': 0.4960649047738277}
Test:       {'precision': 0.5038022813669057, 'recall': 0.5135658914708777, 'f1': 0.5086372310829611, 'auc': 0.6282808774586447, 'prauc': 0.5026892974591488}
Test-subgroups:       {'DIABETES': {'precision': 0.5043804755881804, 'recall': 0.4938725490135555, 'f1': 0.49907120242471414, 'auc': 0.6180418334431103, 'prauc': 0.4992538872530446}, 'HYPERTENSION': {'precision': 0.5058661145582757, 'recall': 0.5194897235966018, 'f1': 0.5125874075847109, 'auc': 0.6316521957376162, 'prauc': 0.506609736168037}, 'CKD': {'precision': 0.4857142857043732, 'recall': 0.5074626865563441, 'f1': 0.4963503599555498, 'auc': 0.621399549059471, 'prauc': 0.47189112526526666}, 'HEART_FAILURE': {'precision': 0.48962148961551133, 'recall': 0.48665048543098727, 'f1': 0.4881314618230758, 'auc': 0.6135713299996752, 'prauc': 0.481

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 588.25it/s]



Epoch: 007, Average Loss: 0.4667
Validation: {'precision': 0.5126582278457836, 'recall': 0.4557877813486504, 'f1': 0.48255318650455054, 'auc': 0.6305749333167654, 'prauc': 0.49587501337028217}
Test:       {'precision': 0.5151245551578509, 'recall': 0.4488372093005859, 'f1': 0.47970173487252626, 'auc': 0.6319890040667921, 'prauc': 0.5086242911331513}
Test-subgroups:       {'DIABETES': {'precision': 0.5119214586183461, 'recall': 0.45117428924040576, 'f1': 0.4796320580884911, 'auc': 0.6220071287061651, 'prauc': 0.5031011366441811}, 'HYPERTENSION': {'precision': 0.5069843878347824, 'recall': 0.44742567077268003, 'f1': 0.47534668222690296, 'auc': 0.6437065061887615, 'prauc': 0.5114167857596852}, 'CKD': {'precision': 0.4746543778692476, 'recall': 0.4449244060379066, 'f1': 0.45930880212987923, 'auc': 0.6198088684791242, 'prauc': 0.47921549331753277}, 'HEART_FAILURE': {'precision': 0.5136298421734056, 'recall': 0.44034440343861814, 'f1': 0.47417218045369064, 'auc': 0.6390744437684643, 'prauc'

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 585.84it/s]



Epoch: 008, Average Loss: 0.4065
Validation: {'precision': 0.5038461538440007, 'recall': 0.4738745980688349, 'f1': 0.48840098920317243, 'auc': 0.6279760638252602, 'prauc': 0.48651099704481215}
Test:       {'precision': 0.5046539862383462, 'recall': 0.48333333333145995, 'f1': 0.49376360616647913, 'auc': 0.6257357980310324, 'prauc': 0.4964457666461959}
Test-subgroups:       {'DIABETES': {'precision': 0.5065616797833784, 'recall': 0.48860759493052397, 'f1': 0.4974226754075885, 'auc': 0.63560406513602, 'prauc': 0.5124962168328739}, 'HYPERTENSION': {'precision': 0.5043668122234035, 'recall': 0.5007225433489833, 'f1': 0.5025380660624115, 'auc': 0.6311343723603359, 'prauc': 0.4921961640124852}, 'CKD': {'precision': 0.47580645160331037, 'recall': 0.5042735042627292, 'f1': 0.4896265510106576, 'auc': 0.617562234365513, 'prauc': 0.48261790514126757}, 'HEART_FAILURE': {'precision': 0.4974747474684662, 'recall': 0.48522167487087164, 'f1': 0.49127181544352955, 'auc': 0.6170195413173262, 'prauc': 0.

In [20]:
def topk_avg_performance_formatted(
    performances,
    long_seq_performances,
    subgroup_performances=None,
    k=5,
):
    """
    根据 overall 指标自动选 top-k 实验，并在这 k 个实验上计算：
      - overall 指标的均值 / 标准差
      - long-sequence 指标的均值 / 标准差
      - （可选）各 subgroup 指标的均值 / 标准差

    参数
    ----
    performances : list[dict]
        每个实验在“总体人群”上的指标，例如：
        [{"f1": 0.8, "auc": 0.9, "prauc": 0.7}, ...]
    long_seq_performances : list[dict]
        每个实验在 long-sequence 人群上的指标，长度与 performances 相同。
    subgroup_performances : list[dict[str, dict]] or None, 默认 None
        若不为 None，则形式为：
            [
                {
                    "DIABETES":     {"f1":..., "auc":..., "prauc":..., ...},
                    "HYPERTENSION": {...},
                    ...
                },
                {
                    "DIABETES":     {...},
                    "HYPERTENSION": {...},
                    ...
                },
                ...
            ]
        外层 list 长度 = 实验数 = len(performances)，
        每个 dict 的 key 为 subgroup 名（如 DIABETES），
        value 为该实验在该 subgroup 上的一组指标。
    k : int
        选取的 top-k 实验数量。

    返回
    ----
    results : dict
        {
            "overall_mean": {...},
            "overall_std": {...},
            "long_seq_mean": {...},
            "long_seq_std": {...},
            "subgroup": {
                subgroup_name: {
                    "mean": {...},
                    "std": {...}
                },
                ...
            } or None,
            "topk_idx": np.ndarray
        }
    """

    n = len(performances)
    if n == 0:
        raise ValueError("performances 为空")

    if len(long_seq_performances) != n:
        raise ValueError("long_seq_performances 长度与 performances 不一致")

    # =======================
    # 1. 根据 overall 选 top-k
    # =======================
    metrics_for_rank = ["f1", "auc", "prauc"]
    scores = {m: np.array([p[m] for p in performances]) for m in metrics_for_rank}
    # 越大越靠前：先按降序排序得到索引，再对索引排序得到名次（从 1 开始）
    ranks = {m: (-scores[m]).argsort().argsort() + 1 for m in metrics_for_rank}
    avg_ranks = np.mean(np.stack([ranks[m] for m in metrics_for_rank], axis=1), axis=1)
    topk_idx = np.argsort(avg_ranks)[:k]

    # =======================
    # 2. overall 均值 / 标准差
    # =======================
    metric_keys = list(performances[0].keys())

    overall_mean = {
        m: np.mean([performances[i][m] for i in topk_idx])
        for m in metric_keys
    }
    overall_std = {
        m: np.std([performances[i][m] for i in topk_idx], ddof=0)
        for m in metric_keys
    }

    # =======================
    # 3. long-seq 均值 / 标准差
    # =======================
    long_metric_keys = list(long_seq_performances[0].keys())
    long_seq_mean = {
        m: np.mean([long_seq_performances[i][m] for i in topk_idx])
        for m in long_metric_keys
    }
    long_seq_std = {
        m: np.std([long_seq_performances[i][m] for i in topk_idx], ddof=0)
        for m in long_metric_keys
    }

    # =======================
    # 4. subgroup（若提供）
    # =======================
    subgroup_results = None
    if subgroup_performances is not None:
        if len(subgroup_performances) != n:
            raise ValueError(
                f"subgroup_performances 长度 {len(subgroup_performances)} "
                f"与 performances 数量 {n} 不一致"
            )

        subgroup_results = {}
        # 从第一个实验的 dict 里拿到 subgroup 名称列表
        subgroup_names = list(subgroup_performances[0].keys())

        for subgroup_name in subgroup_names:
            # 取该 subgroup 对应的 metric dict 列表（按实验索引）
            sub_metric_dicts = [subgroup_performances[i][subgroup_name] for i in topk_idx]

            sub_metric_keys = list(sub_metric_dicts[0].keys())
            sub_mean = {
                m: np.mean([d[m] for d in sub_metric_dicts])
                for m in sub_metric_keys
            }
            sub_std = {
                m: np.std([d[m] for d in sub_metric_dicts], ddof=0)
                for m in sub_metric_keys
            }
            subgroup_results[subgroup_name] = {"mean": sub_mean, "std": sub_std}

    # =======================
    # 5. 打印结果
    # =======================
    print("=== Overall (Top-k) ===")
    for m in overall_mean.keys():
        print(f"{m}: {overall_mean[m]:.4f} ± {overall_std[m]:.4f}")

    print("\n=== Long-sequence (Top-k) ===")
    for m in long_seq_mean.keys():
        print(f"{m}: {long_seq_mean[m]:.4f} ± {long_seq_std[m]:.4f}")

    if subgroup_results is not None:
        print("\n=== Subgroup (Top-k) ===")
        for subgroup_name, res in subgroup_results.items():
            print(f"\n[{subgroup_name}]")
            for m in res["mean"].keys():
                print(f"{m}: {res['mean'][m]:.4f} ± {res['std'][m]:.4f}")

In [21]:
def print_per_class_performance(dfs, col_name="prauc"):
    """
    输入一个 DataFrame 列表，对每个疾病在所有表格的指定列计算 mean ± std 并打印。

    参数:
        dfs (list[pd.DataFrame]): 多个表格组成的列表
        col_name (str): 要计算的指标列名 (默认: "prauc")
    """
    # 拼接所有表格
    all_values = pd.concat(dfs, axis=0)

    # 按疾病分组，计算 mean 和 std
    grouped = all_values.groupby(all_values.index)[col_name].agg(["mean", "std"])

    # 打印
    for disease, row in grouped.iterrows():
        mean_val = row["mean"] * 100
        std_val = row["std"] * 100
        print(f"{disease}: {mean_val:.2f} ± {std_val:.2f}")

In [22]:
if task_type == "binary":
    topk_avg_performance_formatted(final_metrics, final_long_seq_metrics, final_subgroup_metrics)
else:
    final_metrics_global = [metrics["global"] for metrics in final_metrics]
    final_metrics_per_class = [metrics["per_class"] for metrics in final_metrics]
    final_long_seq_metrics_global = [metrics["global"] for metrics in final_long_seq_metrics]
    final_long_seq_metrics_per_class = [metrics["per_class"] for metrics in final_long_seq_metrics]
    topk_avg_performance_formatted(final_metrics_global, final_long_seq_metrics_global)
    print("\nPer-class performance, all patients:")
    print_per_class_performance(final_metrics_per_class, col_name="prauc")
    print("\nPer-class performance, long seq:")
    print_per_class_performance(final_long_seq_metrics_per_class, col_name="prauc")

=== Overall (Top-k) ===
precision: 0.5141 ± 0.0089
recall: 0.5840 ± 0.0625
f1: 0.5452 ± 0.0248
auc: 0.6472 ± 0.0094
prauc: 0.5260 ± 0.0118

=== Long-sequence (Top-k) ===
precision: 0.5126 ± 0.0191
recall: 0.5842 ± 0.0504
f1: 0.5444 ± 0.0161
auc: 0.6391 ± 0.0118
prauc: 0.5182 ± 0.0232

=== Subgroup (Top-k) ===

[DIABETES]
precision: 0.5145 ± 0.0112
recall: 0.5818 ± 0.0609
f1: 0.5444 ± 0.0238
auc: 0.6432 ± 0.0040
prauc: 0.5314 ± 0.0112

[HYPERTENSION]
precision: 0.5072 ± 0.0166
recall: 0.5827 ± 0.0621
f1: 0.5404 ± 0.0228
auc: 0.6438 ± 0.0150
prauc: 0.5271 ± 0.0179

[CKD]
precision: 0.5094 ± 0.0322
recall: 0.5935 ± 0.0580
f1: 0.5448 ± 0.0076
auc: 0.6435 ± 0.0103
prauc: 0.5170 ± 0.0229

[HEART_FAILURE]
precision: 0.5154 ± 0.0154
recall: 0.5827 ± 0.0576
f1: 0.5453 ± 0.0226
auc: 0.6438 ± 0.0133
prauc: 0.5261 ± 0.0169

[CAD]
precision: 0.5095 ± 0.0189
recall: 0.5791 ± 0.0501
f1: 0.5406 ± 0.0199
auc: 0.6404 ± 0.0170
prauc: 0.5218 ± 0.0225

[COPD]
precision: 0.5220 ± 0.0267
recall: 0.5701 ± 0.0